In [18]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    'Akira is a really good chatter <|endoftext|> He is an OnlyFans legend and nnsequentiallover'    
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)
strings = tokenizer.decode(integers)
print(strings)

[33901, 8704, 318, 257, 1107, 922, 37303, 220, 50256, 679, 318, 281, 5514, 36570, 8177, 290, 299, 77, 44399, 72, 439, 2502]
Akira is a really good chatter <|endoftext|> He is an OnlyFans legend and nnsequentiallover


In [7]:
with open('data/the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()
    
enc_text = tokenizer.encode(raw_text)
print(len(enc_text)) 

5145


In [17]:
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f'x: {x}')
print(f'y:      {y}')
print()

for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f'context: {tokenizer.decode(context)} -> {tokenizer.decode([desired])}')

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]

context:  and ->  established
context:  and established ->  himself
context:  and established himself ->  in
context:  and established himself in ->  a


In [20]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(txt)
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [23]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    
    tokenizer = tiktoken.get_encoding("gpt2")
    
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            drop_last=drop_last, num_workers=num_workers)
    
    return dataloader

In [35]:
with open('data/the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()
    
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4,
                                  stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [38]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f'Token ids:\n{inputs}')
print(f'Inputs shape: \n{inputs.shape}')

Token ids:
tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Inputs shape: 
torch.Size([8, 4])


In [39]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [41]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([4, 256])
torch.Size([8, 4, 256])
